# MOTEL Record Builder

This notebook creates, validates, and exports **technology parameter records** according to the schema defined in `data_structure.yaml`.

## Workflow
1. Load and inspect the schema
2. Fill in record fields using Python dicts or helper functions
3. Validate using Pydantic models
4. Export records to JSON (or CSV)

### Schema layers
| Layer | Entities |
|---|---|
| **Main** | `record` |
| **Secondary** | `tech`, `review`, `source` |
| **Supplementary** | `contributor`, `process` |
| **Controlled vocabulary** | `attribute`, `carrier`, `geographic_scope`, `temporal_scope`, `capacity_scope`, `system_boundary`, `uncertainty` |

In [27]:
import importlib
import json
from datetime import date
from pathlib import Path

import motel_record_builder as mrb

# Reload so notebook always picks up latest edits in motel_record_builder.py
mrb = importlib.reload(mrb)

SCHEMA_PATH = Path("data_structure.yaml")

Contributor = mrb.Contributor
Record = mrb.Record
Source = mrb.Source
Tech = mrb.Tech
MotelDataStore = mrb.MotelDataStore
check_record_dependencies = mrb.check_record_dependencies
create_record_if_ready = mrb.create_record_if_ready
export_data_store_csv = mrb.export_data_store_csv
export_dataset_csv_templates = mrb.export_dataset_csv_templates
export_records_csv = mrb.export_records_csv
export_records_jsonl = mrb.export_records_jsonl
load_data_store = mrb.load_data_store
parse_schema_entities = mrb.parse_schema_entities
records_to_dataframe = mrb.records_to_dataframe
register_scope_value = mrb.register_scope_value
register_source = mrb.register_source
register_tech = mrb.register_tech

print("Dependencies and module imports loaded.")

Dependencies and module imports loaded.


In [11]:
# ---------------------------------------------------------------------------
# Load schema entities from helper module
# ---------------------------------------------------------------------------
sections = parse_schema_entities(SCHEMA_PATH)

print("Schema entities loaded:")
for entity, layer in sections.items():
    print(f"  [{layer:15s}]  {entity}")

Schema entities loaded:
  [records        ]  record
  [secondary      ]  tech
  [secondary      ]  review
  [secondary      ]  source
  [supplementary  ]  contributor
  [supplementary  ]  process
  [predefined     ]  attribute
  [predefined     ]  carrier
  [predefined     ]  geographic_scope
  [predefined     ]  temporal_scope
  [predefined     ]  capacity_scope
  [predefined     ]  system_boundary
  [predefined     ]  uncertainty


## Pydantic Models

Typed models map 1-to-1 to the schema.  
Use them to construct records with autocomplete and get instant validation errors.

In [12]:
# ---------------------------------------------------------------------------
# Models and helper functions are now in motel_record_builder.py
# ---------------------------------------------------------------------------
print("Using models/helpers from motel_record_builder.py")

Using models/helpers from motel_record_builder.py


## Helper: `create_record()`

Convenience function that wraps the `Record` constructor, auto-generates a `record_id` if omitted, and defaults `version` dates to today.

In [13]:
# ---------------------------------------------------------------------------
# Record creation helper is imported from motel_record_builder.py
# ---------------------------------------------------------------------------
print("create_record() imported from module.")

create_record() imported from module.


## Workflow Example (5 Steps)

1. Load all existing secondary/supplementary datasets into one object (`MotelDataStore`).
2. Define a draft record payload.
3. Run checks + suggestions to see what exists and what must be created.
4. Optionally create/register missing entities.
5. Add record only when checks pass.

In [28]:
# Step 1: Load existing datasets into one in-memory object
DATASETS_DIR = Path("output") / "datasets"
DATASETS_DIR.mkdir(parents=True, exist_ok=True)

# Keep template files available (safe if already present)
export_dataset_csv_templates(SCHEMA_PATH, DATASETS_DIR)

data_store = load_data_store(DATASETS_DIR)

print("Loaded dataset object summary:")
print(f"  tech: {len(data_store.tech)}")
print(f"  source: {len(data_store.source)}")
print(f"  contributor: {len(data_store.contributor)}")
print(f"  process: {len(data_store.process)}")
print(f"  geographic_scope: {len(data_store.geographic_scope)}")
print(f"  temporal_scope: {len(data_store.temporal_scope)}")
print(f"  capacity_scope: {len(data_store.capacity_scope)}")
print(f"  system_boundary: {len(data_store.system_boundary)}")

# Step 2: User defines a draft record payload
record_draft = {
    "tech_id": "TECH_SOLAR_PV_UTILITY",
    "source_id": "SRC_NREL_ATB_2024",
    "geographic_scope": "GEO_USA",
    "temporal_scope": "TIME_2030",
    "capacity_scope": "CAP_UTILITY",
    "system_boundary": "COND_ISO_BASELOAD",
    "values": [
        {
            "attribute_id": "ATTR_CAPEX",
            "value": 920.0,
            "value_format": "float",
            "value_type": "numeric",
            "unit": "USD/kW",
            "uncertainty": {"min": 780.0, "max": 1100.0},
            "note": "Utility-scale, moderate resource, 2030 projection",
        },
        {
            "attribute_id": "ATTR_EFFICIENCY",
            "value": 0.21,
            "value_format": "float",
            "value_type": "numeric",
            "unit": "fraction",
            "uncertainty": 0.01,
            "note": "Module efficiency (DC), monocrystalline silicon",
        },
        {
            "attribute_id": "ATTR_LIFETIME",
            "value": 30,
            "value_format": "int",
            "value_type": "numeric",
            "unit": "years",
            "uncertainty": None,
            "note": None,
        },
    ],
}

# Step 3: Run check + suggestions
report = check_record_dependencies(draft=record_draft, store=data_store)
check_df = report.to_dataframe()
print(f"Ready to add record? {report.ready}")
check_df

Loaded dataset object summary:
  tech: 0
  source: 0
  contributor: 0
  process: 0
  geographic_scope: 0
  temporal_scope: 0
  capacity_scope: 0
  system_boundary: 0
Ready to add record? False


,field,value,dataset,exists,suggestion
0,tech_id,TECH_SOLAR_PV_UTILITY,tech,False,create new tech entry for 'TECH_SOLAR_PV_UTILITY'
1,source_id,SRC_NREL_ATB_2024,source,False,create new source entry for 'SRC_NREL_ATB_2024'
2,geographic_scope,GEO_USA,geographic_scope,False,create new geographic_scope entry for 'GEO_USA'
3,temporal_scope,TIME_2030,temporal_scope,False,create new temporal_scope entry for 'TIME_2030'
4,capacity_scope,CAP_UTILITY,capacity_scope,False,create new capacity_scope entry for 'CAP_UTILITY'
5,system_boundary,COND_ISO_BASELOAD,system_boundary,False,create new system_boundary entry for 'COND_ISO...


## Export Records

Export one or many records to a JSON file (suitable for loading into the database later).  
An optional helper also flattens records into a pandas DataFrame for quick inspection.

In [29]:
import pandas as pd

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

# Step 4 (optional): register missing entities suggested by the report
if not report.ready:
    print("Missing dependencies found. Registering required entities...")

    register_tech(
        data_store,
        Tech(
            tech_id="TECH_SOLAR_PV_UTILITY",
            technology_name="Solar PV Utility-Scale",
            ehubx_tech_id="solar_pv_util",
            ontology_iri="https://openenergy-platform.org/ontology/oeo/OEO_00010013",
        ),
    )

    register_source(
        data_store,
        Source(
            source_id="SRC_NREL_ATB_2024",
            source_description="NREL Annual Technology Baseline 2024",
            source_type="database",
            link="https://atb.nrel.gov/",
            access_date=date(2024, 6, 1),
            confidence_level="high",
            assessment_method="modeled",
            assessment_date=date.today(),
        ),
    )

    register_scope_value(data_store, "geographic_scope", "GEO_USA")
    register_scope_value(data_store, "temporal_scope", "TIME_2030")
    register_scope_value(data_store, "capacity_scope", "CAP_UTILITY")
    register_scope_value(data_store, "system_boundary", "COND_ISO_BASELOAD")

    report = check_record_dependencies(draft=record_draft, store=data_store)
    print(f"Ready after optional entity creation? {report.ready}")
    display(report.to_dataframe())

# Step 5: add record only if all checks pass
solar_record = create_record_if_ready(draft=record_draft, store=data_store)
solar_record.display()

# Persist updated datasets (secondary/supplementary/scope)
written_dataset_files = export_data_store_csv(data_store, OUTPUT_DIR / "datasets")

# Export records
all_records = data_store.records
jsonl_path = export_records_jsonl(all_records, OUTPUT_DIR / "records.jsonl")
csv_path = export_records_csv(all_records, OUTPUT_DIR / "records.csv")

print(f"Exported JSONL: {jsonl_path}")
print(f"Exported records CSV: {csv_path}")
print("Updated dataset files:")
for p in written_dataset_files:
    print(f"  - {p}")

# Quick preview
df = records_to_dataframe(all_records)
df

Missing dependencies found. Registering required entities...
Ready after optional entity creation? True


,field,value,dataset,exists,suggestion
0,tech_id,TECH_SOLAR_PV_UTILITY,tech,True,existing
1,source_id,SRC_NREL_ATB_2024,source,True,existing
2,geographic_scope,GEO_USA,geographic_scope,True,existing
3,temporal_scope,TIME_2030,temporal_scope,True,existing
4,capacity_scope,CAP_UTILITY,capacity_scope,True,existing
5,system_boundary,COND_ISO_BASELOAD,system_boundary,True,existing


{'assumption_id': None,
 'record_id': 'REC_001',
 'scope': {'capacity_scope': 'CAP_UTILITY',
           'geographic_scope': 'GEO_USA',
           'system_boundary': 'COND_ISO_BASELOAD',
           'temporal_scope': 'TIME_2030'},
 'source_id': 'SRC_NREL_ATB_2024',
 'tech_id': 'TECH_SOLAR_PV_UTILITY',
 'values': [{'attribute_id': 'ATTR_CAPEX',
             'note': 'Utility-scale, moderate resource, 2030 projection',
             'uncertainty': {'max': 1100.0, 'min': 780.0},
             'unit': 'USD/kW',
             'value': 920.0,
             'value_format': 'float',
             'value_type': 'numeric'},
            {'attribute_id': 'ATTR_EFFICIENCY',
             'note': 'Module efficiency (DC), monocrystalline silicon',
             'uncertainty': 0.01,
             'unit': 'fraction',
             'value': 0.21,
             'value_format': 'float',
             'value_type': 'numeric'},
            {'attribute_id': 'ATTR_LIFETIME',
             'note': None,
             'uncerta

,record_id,version_number,date_created,date_modified,tech_id,source_id,assumption_id,geographic_scope,temporal_scope,capacity_scope,system_boundary,attribute_id,value,value_format,value_type,unit,uncertainty,note
0,REC_001,1.0,2026-05-29,2026-05-29,TECH_SOLAR_PV_UTILITY,SRC_NREL_ATB_2024,None,GEO_USA,TIME_2030,CAP_UTILITY,COND_ISO_BASELOAD,ATTR_CAPEX,920.00,float,numeric,USD/kW,"{'min': 780.0, 'max': 1100.0}","Utility-scale, moderate resource, 2030 projection"
1,REC_001,1.0,2026-05-29,2026-05-29,TECH_SOLAR_PV_UTILITY,SRC_NREL_ATB_2024,None,GEO_USA,TIME_2030,CAP_UTILITY,COND_ISO_BASELOAD,ATTR_EFFICIENCY,0.21,float,numeric,fraction,0.01,"Module efficiency (DC), monocrystalline silicon"
2,REC_001,1.0,2026-05-29,2026-05-29,TECH_SOLAR_PV_UTILITY,SRC_NREL_ATB_2024,None,GEO_USA,TIME_2030,CAP_UTILITY,COND_ISO_BASELOAD,ATTR_LIFETIME,30.00,int,numeric,years,None,NaN
